# Langfuse for Observability & Debugging

This tutorial demonstrates how to use Langfuse for comprehensive observability and debugging of LangChain applications. Langfuse provides powerful tracing, monitoring, and evaluation capabilities for LLM applications.

## Table of Contents
1. Setup and Installation
2. Environment Configuration
3. Prompt Management Setup
4. Basic Chain Implementation
5. Agent Implementation with Tools
6. Custom Trace Names
7. Adding Evaluation Scores
8. Advance Monitoring and Performance Optimization

## 1. Setup and Installation

First, we need to install the required packages for this tutorial.

In [1]:
# Install LangChain packages
!pip install langchain langchain-openai langchain-community duckduckgo-search --upgrade -q

# Install specific Langfuse version
!pip install langfuse==2.60.9 -q

**What this does:**
- Installs LangChain core packages for building LLM applications
- Installs Langfuse client for observability and tracing
- Includes DuckDuckGo search tool for agent functionality

## 2. Environment Configuration

### 2.1 Setting Up API Keys

Before using Langfuse, you need to create a project in the Langfuse dashboard and obtain your API keys.

In [2]:
from IPython.display import Markdown
from langfuse import Langfuse
import os
import uuid
from google.colab import userdata

# Configuration constants
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')  # Your OpenAI API key
LANGFUSE_PUBLIC_KEY = userdata.get('LANGFUSE_PUBLIC_KEY')  # Get from Langfuse project settings
LANGFUSE_SECRET_KEY = userdata.get('LANGFUSE_SECRET_KEY')  # Get from Langfuse project settings
LANGFUSE_HOST = "https://cloud.langfuse.com"  # Langfuse cloud host

**Steps to get Langfuse keys:**
1. Go to [https://cloud.langfuse.com](https://cloud.langfuse.com)
2. Create a new project
3. Navigate to project settings
4. Copy the public key and secret key

### 2.2 Environment Validation and Client Initialization

In [3]:
def validate_and_initialize_langfuse():
    """
    Validates API keys and initializes the Langfuse client.

    Returns:
        Langfuse: Initialized Langfuse client instance

    Raises:
        ValueError: If required API keys are missing
    """
    # Validate environment variables
    if not OPENAI_API_KEY:
        print("⚠️  Please set OPENAI_API_KEY in your environment")
        return None

    if not LANGFUSE_PUBLIC_KEY or not LANGFUSE_SECRET_KEY:
        print("⚠️  Please set LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY")
        print("📝 Get them from: https://cloud.langfuse.com")
        return None

    # Set environment variables for other packages
    os.environ["LANGFUSE_PUBLIC_KEY"] = LANGFUSE_PUBLIC_KEY
    os.environ["LANGFUSE_SECRET_KEY"] = LANGFUSE_SECRET_KEY
    os.environ["LANGFUSE_HOST"] = LANGFUSE_HOST
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

    # Initialize Langfuse client
    langfuse = Langfuse(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
        host=LANGFUSE_HOST
    )

    print("🔧 Environment configured successfully!")
    return langfuse

# Initialize the client
langfuse = validate_and_initialize_langfuse()

🔧 Environment configured successfully!


**What this function does:**
- Validates that all required API keys are present
- Sets environment variables for package compatibility
- Creates a Langfuse client instance for tracking and observability
- Returns the initialized client or None if validation fails

In [4]:
from langfuse.callback import CallbackHandler

def create_langfuse_handler(user_id="default_user", session_prefix="session",
                           trace_name="default_trace", release="v1.0"):
    """
    Creates a Langfuse callback handler for tracing LangChain operations.

    Args:
        user_id (str): Unique identifier for the user (should be unique per user)
        session_prefix (str): Prefix for session ID generation
        trace_name (str): Name for the trace in Langfuse dashboard
        release (str): Release version tag for filtering traces

    Returns:
        CallbackHandler: Configured Langfuse callback handler
    """
    return CallbackHandler(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
        host=LANGFUSE_HOST,
        user_id=user_id,
        session_id=f"{session_prefix}_{uuid.uuid4().hex[:8]}",
        trace_name=trace_name,
        release=release
    )

# Create initial handler
langfuse_handler = create_langfuse_handler()

ModuleNotFoundError: Please install langchain to use the Langfuse langchain integration: 'pip install langchain'

**Purpose of the CallbackHandler:**
- Captures all LangChain operations (prompts, model calls, outputs)
- Sends telemetry data to Langfuse for visualization
- Enables performance monitoring and debugging
- Groups related operations under sessions and traces

## 3. Prompt Management Setup

Langfuse provides centralized prompt management, allowing you to version, test, and deploy prompts through their dashboard.

### 3.1 Setting Up Prompts in Langfuse Dashboard

Before running the code, create these prompts in your Langfuse dashboard:

**Prompt 1: `company_prompt`**
```
What industry is {company} in? Give a brief one-sentence answer.
```

![Company Prompt](https://drive.google.com/uc?export=view&id=1ycY7tLVfvyFgrbZ5SViW7fyG9-xwco1u)

**Prompt 2: `analysis_prompt`**
```
Analyze the {industry} industry trends in {year}.
Focus on top 3 challenges and opportunities.
Respond in {format} format.
```
![Analysis Prompt](https://drive.google.com/uc?export=view&id=1AMPkrRQ46u6SKCeZfBxgTS_oZwAwYLzF)

### 3.2 Benefits of Prompt Management

**Why use centralized prompt management?**
- **Version Control**: Track changes and rollback if needed
- **A/B Testing**: Test different prompt versions
- **Collaboration**: Team members can update prompts without code changes
- **Production Safety**: Deploy prompt changes without code deployment

## 4. Basic Chain Implementation

### 4.1 Building a Sequential Chain

In [ ]:
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser

"""
Creates a sequential chain that analyzes company industry trends.

The chain performs two steps:
1. Identifies the industry of a given company
2. Analyzes trends in that industry for a specific year
"""

# Retrieve prompts from Langfuse prompt management
# If no version specified, uses the production label
company_prompt = ChatPromptTemplate.from_template(
    langfuse.get_prompt(name="company_prompt").prompt
)

analysis_prompt = ChatPromptTemplate.from_template(
    langfuse.get_prompt(name="analysis_prompt").prompt
)

# Initialize the language model
model = ChatOpenAI(model="gpt-4o-mini-2024-07-18")

# Create the company identification chain
company_chain = company_prompt | model | StrOutputParser()

# Create the analysis chain that uses the company chain output
analysis_chain = (
    {
        "industry": company_chain,  # Output from company_chain becomes industry input
        "year": itemgetter("year"),  # Pass through year from original input
        "format": itemgetter("format")  # Pass through format from original input
    }
    | analysis_prompt
    | model
    | StrOutputParser()
)

**Chain Architecture Explanation:**
- **Sequential Processing**: Company identification → Industry analysis
- **Data Flow**: Company name → Industry → Industry analysis
- **Parallel Input Passing**: Year and format are passed through unchanged
- **Output Parsing**: Raw model output is converted to clean strings

### 4.2 Executing the Chain with Tracing

In [ ]:
# Executes the company analysis chain with Langfuse tracing.

# Create handler for this specific execution
handler = create_langfuse_handler(
    user_id="john_doe",
    trace_name="chain_execution",
    release="experiment_v1.0"
)

# Execute the analysis chain with tracing enabled
result = analysis_chain.invoke(
    {
        "company": "Tesla",
        "year": "2025",
        "format": "bullet points"
    },
    config={"callbacks": [handler]}
)
Markdown(result)

**What happens during execution:**
1. Input is passed to the company_chain first
2. Company_chain identifies Tesla's industry (automotive/EV)
3. Industry result is passed to analysis_chain along with year and format
4. Analysis_chain generates industry trend analysis
5. All operations are traced and sent to Langfuse dashboard

![Traces](https://drive.google.com/uc?export=view&id=1YODTRMpN-iXe7ohacAsg1mHELpF8gEBY)

![Trace Details](https://drive.google.com/uc?export=view&id=1z-i9OtudCUsIP7JHvb7sY3uocsiiNMIp)


## 5. Agent Implementation with Tools

### 5.1 Building an Agent with Search Capability

In [ ]:
from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain_community.tools import DuckDuckGoSearchRun
from langchain import hub

"""
Creates an OpenAI Functions agent with DuckDuckGo search capability.

The agent can:
- Understand when to use search tools
- Execute searches based on user queries
- Synthesize search results into coherent responses
"""

# Initialize components
llm = ChatOpenAI(model="gpt-4o-mini-2024-07-18", temperature=0)
search = DuckDuckGoSearchRun()

# Configure tools available to the agent
tools = [search]

# Load the standard OpenAI functions agent prompt from LangChain Hub
prompt = hub.pull("hwchase17/openai-functions-agent")

# Create the agent
agent = create_openai_functions_agent(llm, tools, prompt)

# Create executor (handles the agent-tool interaction loop)
agent_executor = AgentExecutor(agent=agent, tools=tools)

**Agent Components Explained:**
- **LLM**: Decides when and how to use tools
- **Tools**: External capabilities (search, APIs, etc.)
- **Prompt**: Instructions for the agent's behavior
- **Executor**: Manages the agent-tool interaction loop

### 5.2 Executing Agent Tasks with Tracing

In [ ]:
# Executes an agent search query with comprehensive tracing.

# Create handler with agent-specific configuration
handler = create_langfuse_handler(
    user_id="john_doe",
    trace_name="agent_execution",
    release="experiment_v1.0"
)

# Execute the search agent with tracing
result = agent_executor.invoke(
    {"input": "What's hot news happening right now?"},
    config={"callbacks": [handler]}
)

Markdown(result["output"])

**Agent Execution Flow:**
1. Agent receives user query
2. Determines if search is needed
3. Executes search tool if necessary
4. Processes search results
5. Generates final response
6. All steps are traced in Langfuse

![Traces](https://drive.google.com/uc?export=view&id=1BJ-fThb2KazM_ElnwO_Ddr826_A8jNYj)


![Trace With Timeline View](https://drive.google.com/uc?export=view&id=1QPOeFugoh3icr2btTXVtmmzAiNQjKZlX)

## 6. Custom Trace Names

### 6.1 Improving Observability with Named Components

In [ ]:
"""
Creates the same analysis chain but with custom names for better tracing visibility.

Custom naming helps with:
- Debugging specific components
- Performance analysis
- Understanding execution flow
"""

# Create prompts with custom run names
company_prompt = ChatPromptTemplate.from_template(
    langfuse.get_prompt(name="company_prompt").prompt
).with_config(run_name="Company Brief Prompt")

analysis_prompt = ChatPromptTemplate.from_template(
    langfuse.get_prompt(name="analysis_prompt").prompt
).with_config(run_name="Company Analysis Prompt")

# Initialize model (no custom name needed)
model = ChatOpenAI(model="gpt-4o-mini-2024-07-18")

# Create named output parser
output_parser = StrOutputParser().with_config(
    run_name="Company Analysis Output Parser"
)

# Create named sub-chains
company_chain = (company_prompt | model | output_parser).with_config(
    run_name="Company Brief"
)

analysis_chain = (
    {
        "industry": company_chain,
        "year": itemgetter("year"),
        "format": itemgetter("format")
    }
    | analysis_prompt
    | model
    | output_parser
).with_config(run_name="Company Analysis")

**Benefits of Custom Naming:**
- **Clearer Debugging**: Identify which component is slow or failing
- **Better Documentation**: Self-documenting execution traces
- **Team Collaboration**: Easier for team members to understand traces
- **Performance Monitoring**: Track specific component performance over time

### 6.2 Executing Named Chain

In [ ]:
"""
Executes the named analysis chain with custom trace naming.
"""

handler = create_langfuse_handler(
    user_id="john_doe",
    trace_name="custom_trace_name",
    release="experiment_v1.0"
)

# Execute named chain
result = analysis_chain.invoke(
    {
        "company": "Tesla",
        "year": "2025",
        "format": "bullet points"
    },
    config={"callbacks": [handler]}
)

Markdown(result)

![Custom Trace Name](https://drive.google.com/uc?export=view&id=1L_TdlxknV03aQOkK-e4p_JOdi04wTFTY)

## 7. Adding Evaluation Scores

### 7.1 Implementing Feedback Loop

In [ ]:
"""
Executes chain and enables feedback scoring for continuous improvement.

This pattern allows you to:
- Track execution performance
- Collect user feedback
- Identify successful vs problematic runs
- Build datasets for model improvement
"""

# Generate consistent ID for trace and feedback linking
trace_id = f"session_{uuid.uuid4().hex[:8]}"

# Create handler with predefined trace ID
handler = CallbackHandler(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host=LANGFUSE_HOST,
    user_id="john_doe",
    session_id=trace_id,
    trace_name="feedback_scores",
    release="experiment_v1.0"
)

# Execute chain with specific run_id for feedback linking
result = analysis_chain.invoke(
    {
        "company": "Apple",
        "year": "2025",
        "format": "bullet points"
    },
    config={
        "run_id": trace_id,
        "callbacks": [handler]
    }
)

In [ ]:
# Display result
Markdown(result)

![User Feedback](https://drive.google.com/uc?export=view&id=1Lb_w-qtuRWi4vgJ-qOvK8LZwjXhd4EHE)

In [ ]:
def add_feedback_score(trace_id, score_value, feedback_comment, score_name="user-feedback"):
    """
    Adds feedback score to a specific trace execution.

    Args:
        trace_id (str): The trace ID to score
        score_value (int): Score value (typically 0-1 or 1-5 scale)
        feedback_comment (str): Human-readable feedback
        score_name (str): Name/category of the score

    Returns:
        None
    """

    langfuse_client = Langfuse()

    langfuse_client.score(
        trace_id=trace_id,
        name=score_name,
        value=score_value,
        comment=feedback_comment
    )

    print(f"✅ Feedback added to trace {trace_id}")

In [ ]:
# Add positive feedback
add_feedback_score(
    trace_id=trace_id,
    score_value=1,  # Positive score
    feedback_comment="This was correct, thank you",
    score_name="user-feedback"
)

![Score](https://drive.google.com/uc?export=view&id=1DhyTQxPkglwoDbwYZ9njCVXHclDkICvH)

**Feedback System Benefits:**
- **Quality Monitoring**: Track which outputs are helpful vs problematic
- **Dataset Creation**: Build labeled datasets for model fine-tuning
- **Performance Trends**: Monitor if model performance improves over time
- **User Satisfaction**: Quantify user experience with the application

### 7.2 Advanced Scoring Patterns

In [ ]:
def add_multiple_scores(trace_id, scores_dict):
    """
    Adds multiple feedback scores to a single trace.

    Useful for comprehensive evaluation across different dimensions.

    Args:
        trace_id (str): Target trace ID
        scores_dict (dict): Dictionary of score_name -> (value, comment) pairs

    Example:
        scores = {
            "accuracy": (0.9, "Very accurate information"),
            "helpfulness": (0.8, "Helpful but could be more detailed"),
            "relevance": (1.0, "Perfectly relevant to the query")
        }
    """

    langfuse_client = Langfuse()

    for score_name, (value, comment) in scores_dict.items():
        langfuse_client.score(
            trace_id=trace_id,
            name=score_name,
            value=value,
            comment=comment
        )

    print(f"✅ Added {len(scores_dict)} scores to trace {trace_id}")

# Example of multi-dimensional scoring
comprehensive_scores = {
    "accuracy": (0.9, "Information appears factually correct"),
    "helpfulness": (0.8, "Good analysis but could include more context"),
    "clarity": (0.95, "Very clear and well-structured output"),
    "completeness": (0.7, "Covers main points but missing some details")
}

add_multiple_scores(trace_id, comprehensive_scores)

![Score Details](https://drive.google.com/uc?export=view&id=1GGwkQTOnxKqGcNM_IwuxY2SEIgsL3Myr)

![Scores](https://drive.google.com/uc?export=view&id=1Cy2GmTfHQjDi_fCjBfrQkqbjEn84A2bx)

## Advance Monitoring and Performance Optimization

### Latency Analysis

![](https://drive.google.com/uc?export=view&id=19nkdr3qt6O4K7LN1CKZ5-NO4_AR87hMm)

#### **1. End-to-End Response Times**

We can refer to the **Trace latency percentiles** panel (left panel):

* **p50**: 3.749 seconds
* **p90**: 9.646 seconds
* **p95**: 12.086 seconds
* **p99**: 15.307 seconds

This reflects the end-to-end time required to process a typical LLM generation request. While median latency (p50) is under 4 seconds, there are significant outliers with p99 hitting over 15 seconds. This indicates occasional latency spikes that may impact user experience for a small subset of users.

---

#### **2. Component-Level Breakdowns**

The **Span latency percentiles** panel (right panel) provides insight into key components:

| Component               | p50    | p95    | Notes                                                |
| ----------------------- | ------ | ------ | ---------------------------------------------------- |
| `retrieval`             | 1.152s | 2.321s | Largest contributor; optimize here for biggest gain  |
| `vector-store`          | 0.447s | 1.184s | Moderate cost, could consider caching                |
| `fetch-prompt-from-...` | 0.127s | 0.783s | Occasionally spikes to 3.2s (see p99) — outlier risk |
| `context-encoding`      | 0.102s | 0.167s | Very minimal impact                                  |

From this, we can see **retrieval** and **vector-store** are the heaviest spans in the overall latency chain.

---

#### **3. Percentile Distributions (P50, P95, P99)**

From the **Generation latency percentiles** panel (center panel):

##### For Generation:

* p50: 2.044s
* p95: 10.577s
* p99: 12.833s

##### For Prompt Embedding:

* p50: 0.521s
* p95: 1.445s
* p99: 2.002s

This confirms that LLM generation is the dominant cost, especially with a long tail — consistent with the overall trace latency.

---
![](https://drive.google.com/uc?export=view&id=1TFnWJryTmaXize3_dTbH6q9buXUH5HDP)

![](https://drive.google.com/uc?export=view&id=1UyE7UlWygbzHHZ4wS0ZfiEFhNNrYG-i-)

#### **4. Historical Trend Analysis**

From the **line charts** in images 1 and 2 (p50 and p99 percentile latencies):

* **gpt-4o-mini** shows **high variability**, spiking up to **9s+ (p50)** and **15s+ (p99)** — indicating possible instability or traffic surges.
* **gpt-4.1-nano** and **gpt-4.1 (2025-04-14)** are stable and low-latency (under 1s consistently across all percentiles).
* **text-embedding-ada-002** has **slightly higher latency than nano models**, but remains fairly stable.

This suggests that **gpt-4o-mini** might be under strain or experiencing uneven workload distribution, while **nano** models offer predictable latency for lightweight tasks.

---

#### **Summary of Analysis**

| Analysis Aspect           | Insight                                                               |
| ------------------------- | --------------------------------------------------------------------- |
| End-to-End Response Times | Median 3.7s, but 99th percentile up to 15s — needs mitigation.        |
| Component-Level Breakdown | Retrieval and vector-store are bottlenecks. Optimization opportunity. |
| Percentile Distribution   | Long tail latency in generation. Prompt embedding is fast and stable. |
| Historical Trend          | gpt-4o-mini has inconsistent performance. nano models are stable.     |

### Cost Monitoring


#### **1. Token Usage Tracking per Model**

From the **Model Costs** panel:

| Model                    | Tokens | Cost (USD) |
| ------------------------ | ------ | ---------- |
| `gpt-4o-mini`            | 53.98K | \$0.012517 |
| `text-embedding-ada-002` | 593    | \$0.000059 |

* **gpt-4o-mini** is the dominant contributor both in token usage and cost.
* **text-embedding-ada-002** usage is minimal, likely used for lightweight vector or retrieval tasks.

**Insight**: The system is heavily reliant on `gpt-4o-mini` for generation tasks.

---

#### **2. Cost per Request / Session / User**

From the **Users Dashboard**:

* Each row represents a **user ID**, with columns:

  * **Total Events** (proxy for requests or sessions)
  * **Total Tokens**
  * **Total Cost**
* This allows you to calculate:

  * **Cost per request**: `Total Cost / Total Events`
  * **Cost per user**: direct from the "Total Cost" column

**Insight**:

* For example:

  * User `u-Gt4O1Gr`: 260 events, 1.09M tokens, cost = \$0.19 → \~**\$0.00073 per request**
  * User `u-epciRym`: 422 events, 591.51K tokens, cost = \$0.60 → \~**\$0.00142 per request**
* This shows **variation in cost efficiency per user**.

---

#### **3. Budget Alerts and Limits**

From the available dashboards:

* **There is no explicit alert/limit setting UI**.
* However, cost tracking is available on a **per-model** and **aggregate** basis, which is foundational for budget control.

**Recommendation**:

* Set manual monitoring thresholds based on token consumption (e.g., flag if `gpt-4o-mini` exceeds 100K/day).

---

#### **4. Model Comparison Economics**

Based on cost-efficiency:

| Model                    | Cost per 1K tokens (approx.) | Notes                               |
| ------------------------ | ---------------------------- | ----------------------------------- |
| `gpt-4o-mini`            | \~\$0.00023                  | Good balance of cost & performance  |
| `text-embedding-ada-002` | \~\$0.0001                   | Very cheap, suitable for embeddings |

**Insight**:
If your use case involves retrieval or classification without complex generation, **shifting more load to embedding-based workflows** (e.g., reranking + prompt compression) can reduce cost.

---

#### **Summary of Cost Monitoring Analysis**

| Monitoring Aspect     | Findings                                                                  |
| --------------------- | ------------------------------------------------------------------------- |
| Token Usage per Model | `gpt-4o-mini` is the major contributor in both usage and cost             |
| Cost per Session/User | Varies significantly; some users use up 1M+ tokens      |
| Budget Control        | Manual tracking feasible; no UI for setting budget alerts            |
| Model Cost Comparison | `text-embedding-ada-002` is cheapest; `gpt-4o-mini` balances quality/cost |

## Summary

This tutorial covered the complete implementation of Langfuse for LLM application observability:

### Key Components Implemented:
1. **Environment Setup**: Proper configuration and validation
2. **Prompt Management**: Centralized prompt versioning and deployment
3. **Chain Tracing**: Sequential operation visibility
4. **Agent Monitoring**: Complex agent-tool interaction tracking
5. **Custom Naming**: Improved debugging and team collaboration
6. **Feedback Systems**: Continuous improvement through user feedback

### Best Practices Learned:
- **Always validate API keys** before initializing clients
- **Use descriptive trace names** for better debugging
- **Implement feedback loops** for quality monitoring
- **Leverage prompt management** for safer deployments
- **Monitor both simple chains and complex agents**

### Advanced Monitoring & Performance Optimization

Beyond tracing, Langfuse enables deeper insights into LLM usage and performance through metrics like latency percentiles, event counts, and feedback data.

#### Key Takeaways:

* **Track latency percentiles (P50/P95/P99)** to detect bottlenecks early
* **Identify high-usage users/sessions** to guide quota control and feature adoption
* **Spot churn indicators** such as 1-day-only active sessions for retention strategies
* **Analyze token/event ratio** to detect inefficient prompts or bloated chains
* **Combine feedback and latency** data to evaluate overall user experience
* **Use historical trends** to inform scaling decisions and future optimization

This comprehensive setup provides full visibility into your LLM application's **behavior**, **performance**, and **user satisfaction**, enabling a feedback-informed cycle of improvement and growth.